# AerisPlane Tutorial 01 — Glider Stability Analysis

This notebook demonstrates the full aerodynamic and stability analysis workflow using the **catalog glider** — a 15 m wingspan sailplane with a 16:1 aspect ratio wing and no propulsion system.

Topics covered:
1. Loading a catalog aircraft and inspecting geometry
2. Single-point aerodynamic analysis
3. Geometry visualisation
4. Alpha sweep — lift, drag, and moment curves per component
5. Longitudinal stability — static margin and neutral point
6. Lateral-directional stability — beta sweep, rate sweeps, modes, time responses

In [ ]:
import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Silence the zero-inertia-tensor UserWarning for the catalog aircraft
import warnings
warnings.filterwarnings('ignore', message='.*Inertia tensor.*')

---
## 1. Load the catalog glider

In [ ]:
from aerisplane.catalog import get_aircraft
from aerisplane.core.flight_condition import FlightCondition

glider = get_aircraft('glider')
print(f"Aircraft : {glider.name}")
print(f"Wings    : {[w.name for w in glider.wings]}")
print()
print(f"Reference area   S = {glider.reference_area():.3f} m²")
print(f"Reference span   b = {glider.reference_span():.2f} m")
print(f"Reference chord  c = {glider.reference_chord():.3f} m")
print(f"Aspect ratio    AR = {glider.reference_span()**2 / glider.reference_area():.1f}")

### Wing geometry

In [ ]:
for w in glider.wings:
    print(f"  {w.name:12s}  span = {w.span():.2f} m   AR = {w.aspect_ratio():.1f}")

---
## 2. Define the flight condition

In [ ]:
# Cruise condition: 20 m/s at 500 m MSL, 3° angle of attack
cruise = FlightCondition(
    velocity = 20.0,   # m/s
    altitude = 500.0,  # m MSL
    alpha    = 3.0,    # deg
    beta     = 0.0,    # deg  (symmetric flight)
)
print(cruise)

---
## 3. Geometry visualisation

In [ ]:
from aerisplane.aero import plot_geometry

plot_geometry(glider, style='three_view', show=True)

In [ ]:
plot_geometry(glider, style='wireframe', show=True)

---
## 4. Single-point aerodynamic analysis

In [ ]:
from aerisplane.aero import analyze

result = analyze(glider, cruise, method='aero_buildup')
result.report()

In [ ]:
# Quick numbers
print(f"CL   = {result.CL:.4f}")
print(f"CD   = {result.CD:.5f}")
print(f"L/D  = {result.CL / result.CD:.1f}")
print(f"Cm   = {result.Cm:.4f}  (about aircraft reference point)")

---
## 5. Alpha sweep — lift, drag, and moment curves

In [ ]:
from aerisplane.aero import alpha_sweep

alpha_range = np.linspace(-6, 14, 41)  # deg

sweep = alpha_sweep(
    glider,
    cruise,
    alpha_range,
    method='aero_buildup',
)

print(f"Components: {list(sweep.components.keys())}")
print(f"Total CL range: {sweep.CL.min():.3f} → {sweep.CL.max():.3f}")

### 5.1 Lift curves — each lifting surface on the same axes

In [ ]:
sweep.plot_lift_curves(show=True)

### 5.2 Moment curves — pitching moment per component

In [ ]:
sweep.plot_moment_curves(show=True)

### 5.3 Full 4-panel plot (CL, CD, Cm, polar)

In [ ]:
sweep.plot(show=True)

### 5.4 Best glide ratio

In [ ]:
ld = sweep.CL / sweep.CD
idx = np.argmax(ld)
print(f"Peak L/D = {ld[idx]:.1f}  at α = {sweep.alphas[idx]:.1f}°")
print(f"CL at best L/D = {sweep.CL[idx]:.3f}")
print(f"CD at best L/D = {sweep.CD[idx]:.5f}")

---
## 6. Longitudinal stability

In [ ]:
from aerisplane.stability import analyze as stab_analyze
from aerisplane.weights import analyze as weight_analyze

weight = weight_analyze(glider)
print(f"Total mass : {weight.total_mass:.2f} kg")
print(f"CG         : [{weight.cg[0]:.3f}, {weight.cg[1]:.3f}, {weight.cg[2]:.3f}] m")

In [ ]:
stab = stab_analyze(glider, cruise, weight, aero_method='aero_buildup')
stab.report()

In [ ]:
print(f"CL_alpha  = {stab.CL_alpha:.4f}  1/deg   (lift curve slope)")
print(f"Cm_alpha  = {stab.Cm_alpha:.4f}  1/deg   (pitch stability, should be < 0)")
print(f"NP        = {stab.neutral_point:.3f}  m   (neutral point x-position)")
print(f"SM        = {stab.static_margin * 100:.1f}%  MAC   (static margin)")

> **Note on static margin:** A positive static margin means the neutral point is aft of the CG — the aircraft is statically stable in pitch. For typical gliders 5–15 % MAC is desirable.

---
## 7. Lateral-directional stability

In [ ]:
from aerisplane.stability import lateral_analyze

lat = lateral_analyze(
    glider,
    cruise,
    weight,
    aero_method = 'aero_buildup',
    beta_range  = np.linspace(-20, 20, 41),
    rate_range  = np.linspace(-0.15, 0.15, 21),
    verbose     = False,
)
print('Lateral analysis complete.')
print('Responses computed:', list(lat.responses.keys()))

### 7.1 Structured text report

In [ ]:
print(lat.report())

### 7.2 Beta sweep — directional and dihedral effect

In [ ]:
lat.plot_static_curves(show=True)

### 7.3 Rate sweeps — roll and yaw damping

In [ ]:
lat.plot_rate_curves(show=True)

### 7.4 Eigenvalue (pole) map

In [ ]:
lat.plot_poles(show=True)

### 7.5 Mode shapes

In [ ]:
lat.plot_mode_shapes(show=True)

### 7.6 Time responses

In [ ]:
lat.plot_time_responses(show=True)

### 7.7 Derivatives bar chart

In [ ]:
lat.plot_derivatives_bar(show=True)

---
## 8. Key derivatives at a glance

In [ ]:
d = lat.deriv
rows = [
    ('CL_alpha', d.CL_alpha, '1/deg', 'lift curve slope'),
    ('Cm_alpha', d.Cm_alpha, '1/deg', 'pitch stability  (want < 0)'),
    ('CY_beta',  d.CY_beta,  '1/deg', 'side force / sideslip'),
    ('Cl_beta',  d.Cl_beta,  '1/deg', 'dihedral effect  (want < 0)'),
    ('Cn_beta',  d.Cn_beta,  '1/deg', 'weathercock      (want > 0)'),
    ('Cl_p',     d.Cl_p,     '1/rad', 'roll damping     (want < 0)'),
    ('Cn_r',     d.Cn_r,     '1/rad', 'yaw damping      (want < 0)'),
    ('Cl_r',     d.Cl_r,     '1/rad', 'roll due to yaw'),
    ('Cn_p',     d.Cn_p,     '1/rad', 'adverse yaw'),
]

print(f"{'Derivative':<12} {'Value':>10}  {'Units':<8}  Note")
print('-' * 60)
for name, val, units, note in rows:
    if val is not None:
        print(f"{name:<12} {val:+10.5f}  {units:<8}  {note}")

---
## 9. A-matrix

The 4×4 linearised lateral-directional state matrix with states **[β (rad), p (rad/s), r (rad/s), φ (rad)]**.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("A matrix (states: β, p, r, φ):")
print(lat.A)

In [ ]:
eigenvalues = np.linalg.eigvals(lat.A)
print("Eigenvalues:")
for i, ev in enumerate(eigenvalues):
    stability = 'stable' if ev.real < 0 else 'UNSTABLE'
    print(f"  λ{i+1} = {ev.real:+.4f} {ev.imag:+.4f}j   {stability}")

---
## Notes on catalog aircraft limitations

The catalog glider uses a **zero inertia tensor** (the weight module returns only total mass without structural layout). AerisPlane falls back to a geometric estimate:

$$I_{xx} \approx \frac{m (b/2)^2}{4}, \quad I_{zz} \approx \frac{m L^2}{12}$$

This makes the dynamic mode eigenvalues qualitatively meaningful but not quantitatively accurate. For a real design, provide a proper mass/inertia breakdown via `WeightResult` — all derivative signs and sweep shapes remain valid regardless.

Similarly, the catalog mass (from a default weight fraction model) is very low (~0.06 kg for a 15 m glider). In practice you would set the mass to the actual all-up weight and compute inertia from a structural model.